In [72]:
path = "../data_base/bbc_text_cls_adapted.csv"

In [73]:
import pandas as pd
import numpy as np
import random
import math
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import r2_score
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import train_test_split

In [74]:
data = pd.read_csv(filepath_or_buffer = path)

In [76]:
labels = data["labels"].unique()

In [77]:
X = data["text"]
y = data["labels"]

In [78]:
if y.dtype == object or str:
    flag = True
    from sklearn.preprocessing import LabelEncoder
    le = LabelEncoder()
    y = le.fit_transform(y)
    y = pd.DataFrame(y)

In [80]:
columns = data.iloc[:,2:]
columns = columns.apply(le.transform)
columns = columns.apply(lambda linha: list(set(linha)), axis=1)

In [81]:
list_cc = []
list_dc = []

for index,_ in enumerate(columns):
    if len(_) == 1:
        list_cc.append(data.iloc[index])
        # list_index_cc.append(index)
    else:
        list_dc.append(data.iloc[index])
        # list_index_cd(_)

df_cc = pd.DataFrame(list_cc)
df_dc = pd.DataFrame(list_dc)

In [82]:
if df_cc.empty == False:
    y_cc = df_cc['labels']
    X_cc = df_cc['text']
    y_dc = df_dc['labels']
    X_dc = df_dc['text']
else:
    y_dc = df_dc['labels']
    X_dc = df_dc['text']

In [83]:
if df_cc.empty == True:
    X_train, X_test, y_train, y_test = train_test_split(X_dc, y_dc, test_size = 0.33, random_state = 1, stratify = y)
else:
    X_train_cc, X_test_cc, y_train_cc, y_test_cc = train_test_split(X_cc, y_cc, test_size = 0.35, random_state = 1, stratify = y)
    X_train_dc, X_test_dc, y_train_dc, y_test_dc = train_test_split(X_dc, y_dc, test_size = 0.65, random_state = 1, stratify = y)
    
    X_train = pd.concat([X_train_cc,X_train_dc], ignore_index=True)
    X_test = pd.concat([X_test_cc,X_test_dc], ignore_index=True)
    y_train = pd.concat([y_train_cc,y_train_dc], ignore_index=True)
    y_test = pd.concat([y_test_cc,y_test_dc], ignore_index=True)

In [84]:
vectorizer = TfidfVectorizer(stop_words='english')

X_train_transformed = vectorizer.fit_transform(X_train)
X_test_transformed = vectorizer.transform(X_test)

In [85]:
indices_treino = y_train.index
columns = data.iloc[indices_treino, 2:]

In [86]:
columns = columns.apply(le.transform)

In [87]:
matrix = np.array(columns)

In [88]:
matrix = np.array([list(dict.fromkeys(lista)) for lista in matrix], dtype=object)

In [89]:
len_ = 0
for _ in matrix:
    if 1 < len(_):
        len_+=1

In [90]:
labels_trans = le.transform(labels)

In [91]:
individual = random.choices(labels_trans, k=len_)

In [92]:
individual = np.array(individual)

In [93]:
naive_bayes_mult = MultinomialNB()

In [94]:
def removal_function(individual,n_remove):
    # individual = list(individual)
    indexes = []
    length = len(individual)
    num = length-n_remove
    index = random.randint(0,num)
    indexes.append(index)
    indexes.append(index+1)
    indexes.append(index+2)

    return(indexes)

In [95]:
def determinator(tiny_list):
    choice = random.choice(tiny_list)
    return(choice)

In [96]:
def insertion_function(indexes,individual,matrix): 
    valores_transformados = [determinator(matrix[v]) for v in indexes]
    for index, _ in enumerate(indexes):
        individual[_] = valores_transformados[index]
        
    return(individual)
    

In [103]:
def fitness_function(individual,matrix,X_train_transformed,a,b,g):
    matrix_ = matrix.copy()
    df_ = pd.DataFrame()
    cont = 0
    saved_dc_idx = []
    saved_cc_idx = []
    dc_sum = 0
    cc_sum = 0

    for index,values in enumerate(matrix_):
        if len(values) > 1:
            matrix_[index] = individual[cont]
            cont+=1
            saved_dc_idx.append((index,matrix_[index]))
        else:
            saved_cc_idx.append((index,matrix_[index]))

    matrix_ = [item[0] if isinstance(item, list) else item for item in matrix_]

    naive_bayes_mult.fit(X_train_transformed,matrix_)
    naive_bayes_mult.predict(X_train_transformed)
    df_ = naive_bayes_mult.predict_log_proba(X_train_transformed)

    for index,values in saved_dc_idx:
        dc_sum += df_[[index],[values]]

    for index,values in saved_cc_idx:
        cc_sum += df_[[index],[values]]

    cont = np.bincount(matrix_)
    variance = np.var(cont)

    #list_before_sum = [max(i) for i in df_]
    #fitness = sum(list_before_sum)

    fitness = a*cc_sum+b*dc_sum-g*variance
    
    return(fitness)

In [99]:
def implementation_function(individual,n_remove,X_train_transformed,matrix,k,a,b,g):
    best_fitness = float('-inf')
    best_individual = []
    fitness = 0
    
    for _ in range(k):
        indexes = removal_function(individual,n_remove)
        individual = insertion_function(indexes,individual,matrix)
        fitness = fitness_function(individual,matrix,X_train_transformed,a,b,g)
        # print(fitness)
        # print(best_fitness)
        
        if fitness > best_fitness:
            best_individual = individual
            best_fitness = fitness
        else:
            if math.exp(fitness - best_fitness) > random.random():
                best_individual = individual
                best_fitness = fitness

    return(best_fitness,best_individual)

In [100]:
a = 1
b = 1
g = 1

In [104]:
best_fitness,best_individual = implementation_function(individual,3,X_train_transformed,matrix,1000,a,b,g)

C:\Users\pedro\AppData\Local\Temp\ipykernel_11464\557226550.py:17: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  if math.exp(fitness - best_fitness) > random.random():
C:\Users\pedro\AppData\Local\Temp\ipykernel_11464\557226550.py:17: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  if math.exp(fitness - best_fitness) > random.random():
C:\Users\pedro\AppData\Local\Temp\ipykernel_11464\557226550.py:17: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  if math.exp(fitness 

In [105]:
print(best_fitness)
print(best_individual)

[-1784.81055366]
[2 0 1 ... 4 2 2]


In [106]:
matrix_ = matrix.copy()

# df_ = pd.DataFrame()
cont = 0
for index,values in enumerate(matrix_):
    if len(values) > 1:
        matrix_[index] = best_individual[cont]
        cont+=1

matrix_ = [item[0] if isinstance(item, list) else item for item in matrix_]

In [107]:
if flag == True:
    matrix_ = le.inverse_transform(matrix_)
    # y_train = le.inverse_transform(y_train)

In [108]:
confusion_matrix(y_train, matrix_)

array([[81, 59, 68, 68, 66],
       [53, 50, 50, 58, 47],
       [51, 47, 67, 64, 50],
       [66, 74, 68, 66, 68],
       [54, 50, 53, 59, 53]])

In [ ]:
df_

""
